# Lab 02: Comparing Model Responses

In this lab you will send the same prompt to a model under different
configurations and systematically compare the results. This is a
key skill for evaluating AI models in production.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["MODEL_ENDPOINT"],
    api_key=os.environ["MODEL_API_KEY"],
)

MODEL = os.environ["MODEL_NAME"]
print(f"Using model: {MODEL}")

## Experiment 1: Temperature sweep

How does temperature affect answer quality for a factual question
vs. a creative one?

In [ ]:
prompts = {
    "factual": "What year was the Python programming language created?",
    "creative": "Write a haiku about cloud computing.",
}

for label, prompt in prompts.items():
    print(f"\n{'='*60}")
    print(f"Prompt type: {label}")
    print(f"{'='*60}")
    for temp in [0.0, 0.5, 1.0]:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=temp,
            messages=[{"role": "user", "content": prompt}],
        )
        answer = response.choices[0].message.content.strip()
        tokens = response.usage.total_tokens
        print(f"\n  temp={temp} ({tokens} tokens):")
        print(f"  {answer}")

## Experiment 2: Max tokens

Controlling `max_tokens` lets you limit response length, which
matters for cost and latency in production.

In [ ]:
prompt = "Explain the concept of neural networks to a non-technical audience."

for max_tok in [25, 75, 200]:
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tok,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response.choices[0].message.content.strip()
    finish = response.choices[0].finish_reason
    print(f"\nmax_tokens={max_tok} (finish_reason={finish}):")
    print(f"  {answer}\n")

## Experiment 3: System prompt impact

How does changing the system prompt alter the model's behavior
for the exact same user question?

In [ ]:
system_prompts = [
    "You are a university professor. Use formal academic language.",
    "You are a friendly tutor. Use simple language and analogies.",
    "You are a sarcastic comedian. Keep it educational but funny.",
]

question = "What is gradient descent?"

for sys_prompt in system_prompts:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": question},
        ],
    )
    print(f"\nPersona: {sys_prompt[:50]}...")
    print(f"  {response.choices[0].message.content.strip()}\n")
    print("-" * 60)

## Exercise

Design your own comparison experiment:
1. Pick a question relevant to your coursework
2. Choose a parameter to vary (temperature, max_tokens, system prompt)
3. Run at least 3 variations and write a short paragraph about what you observed

In [ ]:
# Your experiment here
